In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10, CIFAR100, MNIST, STL10
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mhnlib.utils as mhn_utils
from math import log, sqrt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
import re
from einops import rearrange
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
#device = torch.device("cpu")

# Misc

In [ ]:
def shift_and_rms(x, eps=1e-12):
    shift = x.mean(dim=0, keepdim=True)
    x_shifted = x - shift

    rms = x_shifted.norm(dim=1).square().mean().sqrt()
    rms = rms.clamp_min(eps)

    x_norm = x_shifted / rms

    return shift, rms, x_norm
@torch.no_grad()
def decode_latents(z, z_mean, z_std, vae_model, vae_scaling, batch_size=0, verbose=False):
    orig_ndim = z.ndim

    if z.ndim == 4:
        B, C, H, W = z.shape
        vae_input = z * z_std + z_mean

    elif z.ndim == 5:
        B1, B2, C, H, W = z.shape
        vae_input = z.reshape(B1 * B2, C, H, W)
        vae_input = vae_input * z_std + z_mean

    else:
        raise ValueError("z must have 4 or 5 dimensions.")

    vae_input = vae_input / vae_scaling

    if batch_size > 0:
        vae_decoded = []

        chunks = torch.split(vae_input, batch_size, dim=0)

        for chunk in tqdm(chunks, disable=not verbose):
            chunk = chunk.to(vae_model.device)
            decoded = vae_model.decode(chunk).sample.cpu()
            vae_decoded.append(decoded)

        vae_decoded = torch.cat(vae_decoded, dim=0)

    else:
        vae_decoded = vae_model.decode(
            vae_input.to(vae_model.device)
        ).sample.cpu()

    if orig_ndim == 5:
        vae_decoded = vae_decoded.view(B1, B2, *vae_decoded.shape[1:])

    return (1 + vae_decoded.clamp(-1, 1)) / 2

def get_ema_data(ema_data_filename = 'datasets/gene_data.csv', print_summary = False, top_n = 20):
    gene_df = pd.read_csv(ema_data_filename)


    # -----------------------------
    # Gene metadata
    # -----------------------------
    
    gene_info = (
        gene_df[["geneId", "gene_symbol"]]
        .drop_duplicates("geneId")
        .set_index("geneId")
    )
    
    # -----------------------------
    # 1. Expression per lineage
    # rows = cell_lineage
    # cols = geneId
    # values = average expression
    # -----------------------------
    
    lineage_expr_raw = pd.pivot_table(
        gene_df,
        index="cell_lineage",
        columns="geneId",
        values="expression",
        aggfunc="mean",
        fill_value=0,
    )
    
    gene_ids = lineage_expr_raw.columns.to_numpy()
    lineage_ids = lineage_expr_raw.index.to_numpy()
    
    lineage_expr = lineage_expr_raw.copy()
    lineage_expr_log = lineage_expr_raw.apply(lambda x: torch.tensor(x.values, dtype=torch.float32).log1p().numpy(), axis=1, result_type="expand")
    lineage_expr_log.index = lineage_expr_raw.index
    lineage_expr_log.columns = lineage_expr_raw.columns
    
    lineage_expressions = torch.tensor(
        lineage_expr_raw.to_numpy(),
        dtype=torch.float32,
    ).log1p()
    
    
    # -----------------------------
    # 2. Expression per sample/cell
    # rows = sampleId
    # cols = geneId
    # values = average expression
    # -----------------------------
    
    cell_expr_raw = pd.pivot_table(
        gene_df,
        index="sampleId",
        columns="geneId",
        values="expression",
        aggfunc="mean",
        fill_value=0,
    )
    
    # same gene order as lineage expression
    cell_expr_raw = cell_expr_raw.reindex(columns=gene_ids, fill_value=0)
    
    sample_ids = cell_expr_raw.index.to_numpy()
    
    cell_expressions = torch.tensor(
        cell_expr_raw.to_numpy(),
        dtype=torch.float32,
    ).log1p()
    
    
    # -----------------------------
    # 3. Metadata per sample/cell
    # -----------------------------
    
    metadata_cols = [
        "sampleId",
        "celltype",
        "cell Type Description",
        "cell_lineage",
        "tissue",
        "treatment",
        "immunophenotype",
        "species",
        "strain",
        "sampleCode",
        "RNA.Id",
        "batch",
        "firstPublished",
        "cellOrder",
    ]
    
    metadata_cols = [c for c in metadata_cols if c in gene_df.columns]
    
    cell_metadata = (
        gene_df[metadata_cols]
        .drop_duplicates("sampleId")
        .set_index("sampleId")
        .loc[cell_expr_raw.index]
    )
    
    cell_lineages = cell_metadata["cell_lineage"].to_numpy()
    cell_types = cell_metadata["celltype"].to_numpy()
    
    
    # -----------------------------
    # 4. Top expressed genes per lineage
    # -----------------------------
    
    def top_genes(row, top_n):
        top = row.sort_values(ascending=False).head(top_n)
    
        out = []
        for gene_id, expr in top.items():
            symbol = gene_info.loc[gene_id, "gene_symbol"] if gene_id in gene_info.index else gene_id
            out.append((gene_id, symbol, expr))
    
        return out
    
    
    top_genes_by_lineage = {
        lineage: top_genes(lineage_expr_raw.loc[lineage], top_n=top_n)
        for lineage in lineage_expr_raw.index
    }
    
    
    # -----------------------------
    # 5. Top expressed genes per sample/cell
    # -----------------------------
    
    top_genes_by_cell = {
        sample_id: top_genes(cell_expr_raw.loc[sample_id], top_n=top_n)
        for sample_id in cell_expr_raw.index
    }
    
    
    # -----------------------------
    # 6. Useful lookup dictionaries
    # -----------------------------
    
    geneId_to_col = {gene_id: i for i, gene_id in enumerate(gene_ids)}
    lineage_to_row = {lineage: i for i, lineage in enumerate(lineage_ids)}
    sampleId_to_row = {sample_id: i for i, sample_id in enumerate(sample_ids)}
    
    if print_summary:
        print("lineage_expressions:", lineage_expressions.shape)
        print("cell_expressions:", cell_expressions.shape)
    
        print("\nExample lineage:")
        example_lineage = lineage_ids[0]
        print(example_lineage)
        print(top_genes_by_lineage[example_lineage][:10])
    
        print("\nExample sample/cell:")
        example_cell = sample_ids[0]
        print(example_cell)
        print(cell_metadata.loc[example_cell])
        print(top_genes_by_cell[example_cell][:10])


    return {"cell_expressions": cell_expressions, 
            "cell_lineages" : cell_lineages, 
            "lineage_expressions": lineage_expressions, 
            "lineage_to_row": lineage_to_row }

# Datasets

In [ ]:
DATASET = "ema"
IS_IMAGE = None
if DATASET == "ema":
    dataset_dict = get_ema_data()
    data = dataset_dict['cell_expressions']
    cell_lineages = dataset_dict['cell_lineages']
    lineage_to_row = dataset_dict['lineage_to_row']
    labels = torch.as_tensor([ lineage_to_row[lineage] for lineage in cell_lineages ])
    label_to_name = { el[1] : el[0] for el in lineage_to_row.items()}
    num_labels = len(lineage_to_row)
    IS_IMAGE = False
else:
    label_to_name = {}
    IMG_SIZE = 64
    tfm = T.Compose([
        T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(IMG_SIZE),
        T.Grayscale(num_output_channels=3),
        T.ToTensor(),
    ])
    if DATASET == "cifar100":
        dataset = CIFAR100(root="datasets/cifar100/", train=True, download=True, transform=tfm)
        test_set = CIFAR100(root="datasets/cifar100/", train=False, download=True, transform=tfm)
    if DATASET == "cifar10":
        dataset = CIFAR10(root="datasets/cifar10/", train=True, download=True, transform=tfm)
        test_set = CIFAR10(root="datasets/cifar10/", train=False, download=True, transform=tfm)
    elif DATASET == "stl10":
        dataset = STL10(root="datasets/stl10/", split="train", download=True, transform=tfm)
        test_set = STL10(root="datasets/stl10/", split="test", download=True, transform=tfm)
    elif DATASET == "mnist":
        dataset = MNIST(root="datasets/mnist/", train=True, download=True, transform=tfm)
        test_set = MNIST(root="datasets/mnist/", train=False, download=True, transform=tfm)

    data = torch.as_tensor(dataset.data, dtype=torch.float32)
    data = (data - data.min()) / (data.max() - data.min())
    data = 2*data - 1
    if DATASET == "stl10":
        labels = torch.as_tensor(dataset.labels, dtype=torch.long)
    elif DATASET == "mnist":
        data = rearrange(data, "N H W -> N 1 H W")  # Rearrange to (N, 1, H, W)
        data = data.repeat(1, 3, 1, 1)  # Repeat the single channel to create 3 channels
        labels = torch.as_tensor(dataset.targets, dtype=torch.long)
    else:
        data = rearrange(data, "N H W C -> N C H W")  # Rearrange to (N, C, H, W)
        labels = torch.as_tensor(dataset.targets, dtype=torch.long)
    num_labels = len(torch.unique(labels))
    
    IS_IMAGE = True

### Autoencoder

Loaded only if dealing with images

In [ ]:
if IS_IMAGE:
    from diffusers import AutoencoderKL
    ae_model = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
    ae_model = ae_model.to(device).eval()
    ae_model.requires_grad_(False)
    ae_scaling = ae_model.config.scaling_factor

### Save/subsample data

In [ ]:
torch.manual_seed(1101252)
num_samples_per_label = 5
if IS_IMAGE and num_samples_per_label > 0:
    subset_data_labels = []
    subset_data_indices = []
    for c in range(num_labels):
        label_indices = torch.where(labels == c)[0]
        selected_indices = label_indices[torch.randperm(len(label_indices))[:num_samples_per_label]]
        subset_data_indices.append(selected_indices)
        subset_data_labels.append(torch.full(size=(num_samples_per_label,), fill_value=c))
    subset_data_indices = torch.cat(subset_data_indices)
    subset_data_labels = torch.cat(subset_data_labels)
    subset = data[subset_data_indices]
    if IS_IMAGE:
        with torch.no_grad():
            latents = (ae_model.encode((subset).to(device)).latent_dist.mode()).cpu()
    else:
        latents = None
    torch.save({'data' : subset,
                'labels' : subset_data_labels,
               'indices' : subset_data_indices,
               'latents' : latents,
               'label_to_name' : label_to_name}, 
               f"paper_results/experiments/{DATASET}_per_label={num_samples_per_label}.pt")
else:
    if IS_IMAGE:
        print("For images, choose a num_samples_per_label first.")
    else:
        torch.save({'data' : data,
                'labels' : labels,
               'indices' : None,
               'latents' : None,
               'label_to_name' : label_to_name}, 
               f"paper_results/experiments/{DATASET}.pt")

# Compute fixed points

In [ ]:
DATASETS = ["ema"]
IMAGE_DATASETS = ["mnist", "stl10", "cifar10", "cifar100"]
NUM_ICS_FACTOR = 10
LOGIT_NOISE_STDS = [0.01, 0.1, 1.0, 5.0, 10.0]
num_iterations_quenched = 10000
num_iterations_annealed = 5000
overwrite = True
skip_quenched = False
skip_annealed = True
for DATASET_NAME in DATASETS:
    if DATASET_NAME in IMAGE_DATASETS:
        betas = torch.logspace(-2,3, steps=500)
    else:
        betas = torch.logspace(-3,4, steps=600)
    data_files = list(Path("paper_results/experiments/").glob(f"{DATASET_NAME}*.pt"))
    data_files = [ data_file for data_file in data_files if not 'weights' in  data_file.stem ]
    for data_file in data_files:
        loaded_data = torch.load(data_file)
        data = loaded_data['data']
        labels = loaded_data['labels']
        indices = loaded_data['indices']
        latents = loaded_data['latents']
        label_to_name = loaded_data['label_to_name']
        base_file_name = data_file.stem + "_weights_%s_centered=%s_euclidean=%s"

        raw_patterns = torch.clone(data.view(data.shape[0],-1)) if latents is None else torch.clone(latents.view(latents.shape[0],-1))
        K = raw_patterns.shape[0]
        NUM_ICS = NUM_ICS_FACTOR*K
        BATCH_SIZE = 1000 #(NUM_ICS_FACTOR//2)*K
        
        w_ics = torch.zeros(NUM_ICS, K)
        w_ics[torch.arange(NUM_ICS), torch.repeat_interleave(torch.arange(0,K), NUM_ICS_FACTOR)] = 1.0
        
        for is_centered in [False, True]:
            for is_euclidean in [False, True]:
                if is_euclidean and is_centered:
                    continue
                if is_centered or is_euclidean:
                    shift, rms, patterns = shift_and_rms(raw_patterns)
                else:
                    shift = torch.zeros(raw_patterns.shape[-1])
                    rms = 1.0
                    patterns = raw_patterns
                
                gram = patterns @ patterns.T
                if is_euclidean:
                    biases = -0.5*torch.diagonal(gram)
                else:
                    biases = torch.zeros(K)
                    
                quenched_file_name = base_file_name % ("quenched", is_centered, is_euclidean) + f".pt"
                quenched_save_path = data_file.parent / quenched_file_name
                run_quenched = (overwrite or ((not overwrite) and (not quenched_save_path.exists())))
                if run_quenched and (not skip_quenched):
                    pbar = tqdm(range(0, NUM_ICS, BATCH_SIZE))
                    pbar.set_description(f"{DATASET_NAME}, quenched. K: {K}.  Centered: {is_centered}. Euclidean: {is_euclidean}. ")
                    torch.manual_seed(1101252)
                    w_quenched = []
                    for batch_start in pbar:
                        torch.cuda.empty_cache()
                        batch_end = min(batch_start + BATCH_SIZE, NUM_ICS)
                        w_quenched_batch = mhn_utils.dual_deterministic_dynamics(gram.to(device), 
                                                                         biases.to(device), 
                                                                         betas.to(device), 
                                                                         w_ics[batch_start:batch_end].to(device), 
                                                                         num_iterations=num_iterations_quenched, 
                                                                         verbose=False).cpu().squeeze()
                        w_quenched.append(w_quenched_batch)
                    w_quenched = torch.cat(w_quenched, dim=1)
    
                    mean_entropy_quenched = mhn_utils.get_entropy(w_quenched).mean(dim=-1)
    
                    print(f"Results. Uniform entropy: {log(K)}. Max/min found entropies {mean_entropy_quenched[0]} {mean_entropy_quenched[-1]}" )
    
                    torch.save({'weights' : w_quenched, 'w_ics' :  w_ics,
                               'biases' : biases, 'betas' : betas, 'patterns' : patterns,
                               'shift' : shift, 'rms' : rms }, quenched_save_path)
                else:
                    print(f"SKIPPING: {DATASET_NAME}, quenched. K: {K}.  Centered: {is_centered}. Euclidean: {is_euclidean}. ")
                
                if skip_annealed:
                    continue
                
                for LOGIT_NOISE_STD in LOGIT_NOISE_STDS:
                    print(f"{DATASET_NAME}, annealed. K: {K}.  Centered: {is_centered}. Euclidean: {is_euclidean}. Logit noise: {LOGIT_NOISE_STD}")
                    
                    annealed_file_name = base_file_name % ("annealed", is_centered, is_euclidean) + f"_noise={LOGIT_NOISE_STD:.2f}.pt"
                    annealed_save_path = data_file.parent / annealed_file_name
                    if not overwrite and annealed_save_path.exists():
                        continue
                    
                    
                    torch.manual_seed(1101252)
                    w_annealed = []
                    for batch_start in range(0, NUM_ICS, BATCH_SIZE):
                        torch.cuda.empty_cache()
                        batch_end = min(batch_start + BATCH_SIZE, NUM_ICS)
                        w_annealed_batch = mhn_utils.dual_deterministic_dynamics_annealing(gram.to(device), 
                                            biases.to(device), 
                                            betas.to(device), 
                                            w_ics[batch_start:batch_end].to(device),
                                            logit_noise_std=LOGIT_NOISE_STD,
                                            num_iterations=num_iterations_annealed,
                                            verbose=True).cpu().squeeze()
                        w_annealed.append(w_annealed_batch)
                    w_annealed = torch.cat(w_annealed, dim=1)


                    torch.save({'weights' : w_annealed, 'w_ics' :  w_ics,
                               'biases' : biases, 'betas' : betas, 'patterns' : patterns,
                               'shift' : shift, 'rms' : rms, 'logit_noise' : LOGIT_NOISE_STD },  annealed_save_path)


                
